In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(bool(os.getenv("ANTHROPIC_API_KEY")))

True


In [19]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [7]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

In [16]:
def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 2000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    for attempt in range(3):
        message = client.messages.create(**params)

        text_blocks = []

        for block in message.content:
            text = getattr(block, "text", None)
            if text:
                text_blocks.append(text)

        if text_blocks:
            return "\n".join(text_blocks)

    return None

In [29]:
def run_prompt(prompt_inputs):
    prompt = f"""
Extract all important topics from the passage below.

Return ONLY a valid JSON array of strings.

STRICT OUTPUT RULES:
1. Do not include markdown.
2. Do not include code fences.
3. Do not include an introduction.
4. Do not include explanations.
5. Do not include commentary.
6. Do not include any text before or after the JSON array.
7. Your entire response must begin with [ and end with ].

<passage>
{prompt_inputs["content"]}
</passage>
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [30]:
test_output = run_prompt(evaluation_dataset[0])

print(test_output)

["Climate change", "Global agricultural systems", "Rising temperatures", "Changing precipitation patterns", "Increased drought frequency", "Crop resilience", "Food security", "Sustainable farming practices"]


In [12]:
sample_input = {
    "content": """
Recent advances in artificial intelligence have transformed healthcare research,
particularly in medical imaging, predictive analytics, and clinical decision support.
Researchers are also examining the ethical implications of AI systems, including
algorithmic bias, patient privacy, and transparency in automated decision-making.
"""
}

output = run_prompt(sample_input)
print(output)

Based on this text, the topics covered include:

**AI Applications in Healthcare:**
1. **Medical imaging** – using AI to analyze scans, X-rays, or other diagnostic images
2. **Predictive analytics** – using AI to forecast patient outcomes or disease risk
3. **Clinical decision support** – AI tools that assist healthcare providers in making treatment decisions

**Ethical Considerations of AI in Healthcare:**
4. **Algorithmic bias** – concerns about AI systems producing unfair or skewed results
5. **Patient privacy** – protecting sensitive health data used by AI systems
6. **Transparency in automated decision-making** – ensuring AI processes are explainable and understandable

Would you like me to expand on any of these topics or help with something related to this text (e.g., summarizing, analyzing, or using it for a specific purpose)?


In [1]:
evaluation_dataset = [
    {
        "content": """
Climate change is altering global agricultural systems through rising temperatures,
changing precipitation patterns, and increased drought frequency. Researchers are studying
crop resilience, food security, and sustainable farming practices as possible responses.
"""
    },
    {
        "content": """
Social media platforms influence adolescent mental health through social comparison,
cyberbullying, and disrupted sleep patterns. Researchers have also examined how online
communities may provide social support and opportunities for identity development.
"""
    },
    {
        "content": """
Renewable energy technologies such as solar and wind power are becoming increasingly
important in efforts to reduce greenhouse gas emissions. Challenges include energy storage,
grid reliability, and the economic costs of transitioning away from fossil fuels.
"""
    },
    {
        "content": """
Advances in genomic medicine have improved researchers' ability to identify genetic risk
factors for disease. Precision medicine, genetic testing, data privacy, and ethical concerns
about discrimination remain important areas of study.
"""
    }
]

In [21]:
import json

def grade_output(test_case, output):
    grading_prompt = f"""
You are evaluating whether an AI response correctly extracts topics from a scholarly passage.

Original passage:
<passage>
{test_case["content"]}
</passage>

AI response:
<response>
{output}
</response>

Evaluation criteria:
1. The response should contain a JSON array of strings.
2. Each string should contain only a topic mentioned in the passage.
3. The response should include the important topics from the passage.
4. The response should contain the JSON array and nothing else.

Return valid JSON only in exactly this format:
{{
    "score": 8,
    "reasoning": "Brief explanation of why the response earned this score."
}}

Use a score from 1 to 10.
"""

    messages = []
    add_user_message(messages, grading_prompt)
    result = chat(messages)

    result = result.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(result)

In [31]:
evaluation_results = []

for test_case in evaluation_dataset:
    output = None

    while output is None:
        output = run_prompt(test_case)

    grade = grade_output(test_case, output)

    evaluation_results.append({
        "content": test_case["content"],
        "output": output,
        "score": grade["score"],
        "reasoning": grade["reasoning"]
    })

print(json.dumps(evaluation_results, indent=2))

[
  {
    "content": "\nClimate change is altering global agricultural systems through rising temperatures,\nchanging precipitation patterns, and increased drought frequency. Researchers are studying\ncrop resilience, food security, and sustainable farming practices as possible responses.\n",
    "output": "[\"Climate change\", \"Global agricultural systems\", \"Rising temperatures\", \"Changing precipitation patterns\", \"Increased drought frequency\", \"Crop resilience\", \"Food security\", \"Sustainable farming practices\"]",
    "score": 10,
    "reasoning": "The response is a valid JSON array of strings containing nothing else. Each topic listed (climate change, global agricultural systems, rising temperatures, changing precipitation patterns, increased drought frequency, crop resilience, food security, sustainable farming practices) is directly mentioned in the passage, and together they comprehensively cover all the important topics from the text without adding extraneous or fab

In [32]:
from statistics import mean

scores = [result["score"] for result in evaluation_results]

average_score = mean(scores)

print("Prompt Evaluation Summary")
print("-------------------------")
print(f"Test cases: {len(scores)}")
print(f"Average score: {average_score:.1f}/10")
print()

for index, result in enumerate(evaluation_results, start=1):
    print(f"Test case {index}: {result['score']}/10")

Prompt Evaluation Summary
-------------------------
Test cases: 4
Average score: 9.8/10

Test case 1: 10/10
Test case 2: 10/10
Test case 3: 9/10
Test case 4: 10/10


In [28]:
print(evaluation_results[0]["reasoning"])

The topics extracted are accurate, complete, and directly reflect the passage's content (climate change, agricultural systems, rising temperatures, precipitation patterns, drought frequency, crop resilience, food security, and sustainable farming practices). However, the response violates criterion 4 by wrapping the JSON array in markdown code fence syntax (```json and ```), rather than providing the JSON array and nothing else. This formatting addition, while common and easily parsed by most systems, technically fails the strict requirement of containing only the JSON array.
